# 🛡️ Cryptographic Attack & Vulnerability Simulation Suite for BAKMM-IoD

This notebook details the security vulnerabilities, active exploits, and defensive limits of the **BAKMM-IoD** protocol under both the **Dolev–Yao (DY)** and **Canetti–Krawczyk (CK)** adversary models [1, 2]. 

Below is the list of five key security scenarios, architectural loopholes, and mathematical flaws to implement in our simulation suite.


## 1. Mathematical Specification Mismatch Bug (Protocol Breakdown)

### 🔍 Vulnerability Description
Before simulating external adversaries, we must model the **mathematical order-sensitivity** of the hash functions. In the BAKMM-IoD paper's specification, there is a mismatch in the parameter concatenation sequence when calculating the Session Key ($SK$) between the Ground Station Server ($ES_j$) and the Drone ($DE_i$).

* **GSS-Side Formula (Step AKDDE2) [3]:**
  $$SK_{ES_j, DE_i} = h(A \parallel B \parallel \mathbf{T_1 \parallel T_2 \parallel RID_{DE_i}} \parallel MS_{DE_i-ES_j})$$
* **Drone-Side Formula (Step AKDDE3) [4]:**
  $$SK_{DE_i, ES_j} = h(A \parallel B \parallel \mathbf{RID_{DE_i} \parallel T_1 \parallel T_2} \parallel MS_{DE_i-ES_j})$$

Where $A = h(TC_{DE_i} \parallel rs_1 \parallel MS_{DE_i-ES_j} \parallel T_1)$ and $B = h(RID_{ES_j} \parallel TC_{ES_j} \parallel rs_2 \parallel MS_{DE_i-ES_j} \parallel T_2)$.

### 🛠️ How to Simulate
1. **Execute Handshake with Literal Formulas:** Program the Drone and GSS to use the exact byte order specified in the paper.
2. **Assertion Test:** Assert that the resulting keys match: `assert drone_SK == gss_SK`.
3. **Outcome:** The assertion will **fail**, proving that the literal protocol specification prevents communication.
4. **Resolution:** Implement a "Corrected State" scenario where both parties agree on a single, uniform concatenation order to enable the subsequent attack tests.


## 2. State Update Desynchronization Attack (Active DoS Lockout)

### 🔍 Vulnerability Description
During Step AKDDE3 of the handshake, the Drone receives the challenge $MSG_2$, verifies GSS's signature $M_4$, decrypts the new temporary identity $TID_{new}$, and **instantly updates** its local storage [4]. It then transmits $MSG_3 = \{M_6, T_3\}$ [5].
The GSS, however, only commits $TID_{new}$ to its active mapping table *after* successfully validating $MSG_3$ in Step AKDDE4 [5, 6].

An active **Dolev-Yao** adversary can exploit this gap by performing a **selective packet drop** [1, 2].

Drone (DE_i)                                         GSS (ES_j) |                                                     | [Decrypts TID_new]                                        | [Updates state locally]                                   | [Sends MSG3]                                              | |------------ X (Attacker Intercepts & Drops) ------->| [GSS retains OLD TID]

### 🛠️ How to Simulate
1. **Initialize Session 1:** Establish a successful $MSG_1 \to MSG_2$ transmission.
2. **Interception Step:** Program the network interface to drop $MSG_3$ so GSS never receives it.
3. **State Check:** Check that the Drone's active identity is now $TID_{new}$ while GSS's map still expects the old $TID$.
4. **Initialize Session 2:** Force the Drone to initiate a new handshake using its new credentials.
5. **Outcome:** GSS throws an `IdentityNotFoundError` and terminates the handshake, permanently locking out the legitimate Drone.



## 3. Timestamp Verification and Replay Attack

### 🔍 Vulnerability Description
The protocol guards against **replay attacks** by validating message freshness using timestamps ($T_x$) and a preset transmission delay threshold ($\Delta T$) [7, 8]:
$$|T_x - T_x^*| \le \Delta T$$
An adversary recording old handshake signals will attempt to play back $MSG_1 = \{TID_{DE_i}, M_1, M_2, T_1\}$ to authenticate unauthorized sessions [1, 9].

### 🛠️ How to Simulate
1. **Eavesdrop:** Run a successful handshake and log $MSG_1$.
2. **Delay:** Sleep the simulation thread for $2 \times \Delta T$.
3. **Replay:** Inject the exact captured $MSG_1$ into the GSS's receiver queue.
4. **Verification Step:** Assert that GSS rejects the packet with a `TimestampExpired` exception.


## 4. Clock Drift / GPS Spoofing Attack (Denial of Service)

### 🔍 Vulnerability Description
While strict timestamp validation prevents replays [8], it introduces a major physical vulnerability. Drones operate in dynamic environments where **GPS spoofing** or **unstable network jitter** can cause clock desynchronization between nodes [10, 11].

### 🛠️ How to Simulate
1. **Configure Drift:** Program the Drone's local clock to drift ahead of GSS by a delta larger than the threshold (e.g., $t_{drift} = 50\text{ ms}$ where $\Delta T = 10\text{ ms}$).
2. **Attempt Handshake:** Initiate a standard, legitimate authentication process from the Drone.
3. **Outcome:** GSS evaluates the fresh timestamp as "invalid/expired" or "from the future" and drops the connection, causing a complete loss of service.


## 5. Ephemeral Secret Leakage (ESL) Attack under the CK-Adversary Model

### 🔍 Vulnerability Description
Under the robust **Canetti-Krawczyk (CK) adversary model**, an attacker has the capability to compromise ephemeral session states, including session-specific random secrets ($rs_1$ or $rs_2$), but *cannot* access long-term secret parameters ($MS_{DE_i-ES_j}$, $TC_{DE_i}$) [1, 2]. A cryptographically secure scheme must still guarantee **Session Key Secrecy** under these conditions [12, 13].

### 🛠️ How to Simulate
1. **Expose Ephemeral Secrets:** Let the adversary intercept and read the active session's random keys $rs_1$ and $rs_2$.
2. **Attempt Key Calculation:** Have the adversary compute the target session key using the leaked values:
   $$SK_{adversary} = h(f(rs_1) \parallel f(rs_2) \parallel \dots)$$
3. **State Isolation Check:** Attempt to compute the full $SK$ without knowing the long-term keys ($TC_{DE_i}$, $MS_{DE_i-ES_j}$).
4. **Outcome:** Prove via a `hash_mismatch` assertion that the adversary's computed session key does not match the actual session key, validating the protocol's CK-resilience [12, 13].